In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv(
    "../prepared_data/reduced_vars_with_hmm.csv",
    index_col=0,
    parse_dates=True
)

In [31]:
df

,Expected_Inflation_5Y__lvl__L52,Fed_Total_Assets__pct1,CPI__pct4,Sector_Rotation__lvl__L4,Japan_10Y_JGB__d26,CPI_YoY__lvl__rollstd52,sell_jewelry__lvl,Industrial_Production__pct52,CPI__pct52,Building_Permits__pct8,...,NFCI__d1,US_2Y_Treasury__lvl__rollstd8,NFCI__d1__z8,y_SP500_bin_4w,prob_state_0,prob_state_1,prob_state_2,regime_entropy,regime_persist,regime_label
2005-04-08,2.34,-0.012661,0.006725,-0.002544,-0.226,0.322931,22.0,0.040065,0.035106,0.017029,...,0.01319,0.142221,0.681459,1.0,0.999929,0.000071,1.551628e-09,0.000753,0.978292,0
2005-04-15,2.44,0.010510,0.006725,-0.002218,-0.226,0.311846,22.0,0.040065,0.035106,0.017029,...,0.01160,0.124183,0.189865,1.0,0.983861,0.016122,1.704377e-05,0.082738,0.978020,0
2005-04-22,2.29,-0.004858,0.006725,-0.014507,-0.226,0.298461,22.0,0.040065,0.035106,0.017029,...,0.00944,0.113358,-0.847980,1.0,0.997830,0.002170,2.581785e-07,0.015478,0.978257,0
2005-04-29,2.34,0.009983,0.000000,0.001789,-0.226,0.282448,22.0,0.040065,0.035106,0.042677,...,0.00706,0.102878,-1.780810,1.0,0.999132,0.000868,1.081338e-07,0.006985,0.978279,0
2005-05-06,2.46,-0.009120,-0.001028,-0.016937,-0.214,0.285433,20.0,0.033535,0.028027,0.011154,...,0.00519,0.102878,-1.692287,1.0,0.995820,0.004180,3.755339e-07,0.027073,0.978222,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-31,2.37,-0.000379,0.000000,-0.030473,0.155,0.253120,82.0,0.019286,0.029500,-0.002827,...,0.00376,0.058797,1.157961,1.0,0.110543,0.889423,3.440337e-05,0.348033,0.963206,1
2025-11-07,2.43,-0.002171,-0.002087,-0.031655,0.305,0.253120,82.0,0.020827,0.027351,-0.019081,...,0.00335,0.058661,0.699818,1.0,0.087036,0.912919,4.463259e-05,0.296113,0.962807,1
2025-11-14,2.39,0.001176,-0.002087,-0.001091,0.305,0.253120,82.0,0.020827,0.027351,-0.019081,...,0.00179,0.063696,-0.719084,1.0,0.019797,0.980162,4.114231e-05,0.097704,0.961667,1
2025-11-21,2.43,-0.003826,-0.002087,0.016741,0.305,0.253120,82.0,0.020827,0.027351,-0.019081,...,0.00047,0.057321,-1.662558,1.0,0.003139,0.996755,1.053645e-04,0.022298,0.961384,1


In [32]:
TARGET_COL = "y_SP500_bin_4w"

y = df[TARGET_COL].copy()

X = df.drop(columns=[TARGET_COL], errors="ignore").copy()
X = X.drop(columns=["SP500"], errors="ignore")  # quita nivel del índice si existe

# Solo numéricas
X = X.select_dtypes(include=[np.number]).copy()

# Limpieza básica
X = X.replace([np.inf, -np.inf], np.nan)

data = X.join(y.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts(dropna=False))

X shape: (1078, 36)
y value counts:
 target
1.0    857
0.0    221
Name: count, dtype: int64


In [33]:
# Make y numeric if binary as strings (IN/OUT)  -> PIPELINE: IN=0, OUT=1
if y.dtype == "object":
    y_mapped = y.map({"IN": 0, "OUT": 1})
    if y_mapped.isna().any():
        raise ValueError(f"Valores inesperados en y: {y.unique()}")
    y = y_mapped.astype(int)
else:
    # bool -> int, float-int -> int
    if y.dtype == "bool":
        y = y.astype(int)
    elif np.issubdtype(y.dtype, np.number):
        if np.all(np.isclose(y.values, y.values.astype(int))):
            y = y.astype(int)

print("y dtype:", y.dtype, "| unique:", np.unique(y))
print("Counts [IN=0, OUT=1]:", np.bincount(y))

y dtype: int64 | unique: [0 1]
Counts [IN=0, OUT=1]: [221 857]


In [34]:
CUTOFF_DATE = "2024-01-01"  # ajusta si tu compañero usa otra

X_train = X.loc[X.index < CUTOFF_DATE].copy()
X_test  = X.loc[X.index >= CUTOFF_DATE].copy()

y_train = y.loc[y.index < CUTOFF_DATE].copy()
y_test  = y.loc[y.index >= CUTOFF_DATE].copy()

print("Train:", X_train.index.min(), "->", X_train.index.max(), "| n =", len(X_train))
print("Test :", X_test.index.min(), "->", X_test.index.max(), "| n =", len(X_test))
print("y_train counts:\n", y_train.value_counts())
print("y_test counts:\n", y_test.value_counts())

Train: 2005-04-08 00:00:00 -> 2023-12-29 00:00:00 | n = 978
Test : 2024-01-05 00:00:00 -> 2025-11-28 00:00:00 | n = 100
y_train counts:
 target
1    771
0    207
Name: count, dtype: int64
y_test counts:
 target
1    86
0    14
Name: count, dtype: int64


### MODEL: CATBOOST

In [35]:
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid

from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    f1_score
)

In [36]:
tscv = TimeSeriesSplit(n_splits=3)


In [37]:
# 0 = DOWN, 1 = UP
counts = y_train.value_counts().sort_index()

if 0 not in counts.index or 1 not in counts.index:
    raise ValueError("y_train debe tener clases 0 y 1. Revisa el encoding.")

w0 = len(y_train) / (2 * counts.loc[0])
w1 = len(y_train) / (2 * counts.loc[1])

w0, w1 = float(w0), float(w1)

print("Class weights implied -> IN(0):", round(w0, 4), "OUT(1):", round(w1, 4))

# sample_weight por fila
sw_all = y_train.map({0: w0, 1: w1}).values
print("sample_weight length:", len(sw_all))

Class weights implied -> IN(0): 2.3623 OUT(1): 0.6342
sample_weight length: 978


In [38]:
param_grid = {
    "iterations": [400, 800],
    "depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "l2_leaf_reg": [10, 30, 50],
    "border_count": [32, 64, 128]
}
# Cuántas combinaciones estás probando
n_combos = len(list(ParameterGrid(param_grid)))
print("Grid combinations:", n_combos)

Grid combinations: 108


In [39]:
best_score = -1
best_params = None

for params in ParameterGrid(param_grid):
    fold_scores = []

    for tr_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        sw_tr = sw_all[tr_idx]

        model = CatBoostClassifier(
            loss_function="Logloss",
            random_seed=42,
            verbose=False,
            **params
        )

        model.fit(
    X_tr, y_tr,
    sample_weight=sw_tr,
    eval_set=(X_val, y_val),
    use_best_model=True,
    early_stopping_rounds=50,
    verbose=False
)

        proba = model.predict_proba(X_val)[:, 1]
        pred = (proba >= 0.5).astype(int)

        fold_scores.append(balanced_accuracy_score(y_val, pred))

    mean_score = float(np.mean(fold_scores))

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("Best CV balanced_accuracy:", round(best_score, 4))
print("Best params:", best_params)

Best CV balanced_accuracy: 0.6291
Best params: {'border_count': 32, 'depth': 3, 'iterations': 400, 'l2_leaf_reg': 30, 'learning_rate': 0.05}


In [40]:
best_cat = CatBoostClassifier(
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
    **best_params
)

best_cat.fit(X_train, y_train, sample_weight=sw_all)

CatBoostClassifier(border_count=32, depth=3, iterations=400, l2_leaf_reg=30, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [41]:
val_ratio = 0.2
split_val = int(len(X_train) * (1 - val_ratio))

X_tr, X_val = X_train.iloc[:split_val], X_train.iloc[split_val:]
y_tr, y_val = y_train.iloc[:split_val], y_train.iloc[split_val:]

sw_tr = y_tr.map({0: w0, 1: w1}).values

tmp = CatBoostClassifier(
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
    **best_params
)
tmp.fit(X_tr, y_tr, sample_weight=sw_tr)

val_proba = tmp.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.35, 0.60, 51)

best_thr = 0.5
best_f1_down = -1

for thr in thresholds:
    val_pred = (val_proba >= thr).astype(int)
    f1_down = f1_score(y_val, val_pred, pos_label=0)
    if f1_down > best_f1_down:
        best_f1_down = f1_down
        best_thr = float(thr)

print("Best threshold:", round(best_thr, 3), "| Best F1 DOWN:", round(best_f1_down, 4))

Best threshold: 0.47 | Best F1 DOWN: 0.56


In [42]:
# =====================================================
# STEP 6 — Test analytics (2024 onward) — BINARY
# Model: CatBoost (best_cat)
# 0 = OUT, 1 = IN
# =====================================================
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    roc_auc_score
)

# Refit final en todo train
best_cat.fit(X_train, y_train, sample_weight=sw_all)

# TEST (2024+)
test_proba = best_cat.predict_proba(X_test)[:, 1]   # p(IN=1)
test_pred  = (test_proba >= best_thr).astype(int)

print("\n=== CatBoost (TEST) ===")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Balanced Acc:", round(balanced_accuracy_score(y_test, test_pred), 4))
print("F1 (IN):", round(f1_score(y_test, test_pred, pos_label=1), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True OUT (0)", "True IN (1)"],
    columns=["Pred OUT (0)", "Pred IN (1)"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test, test_pred, target_names=["OUT (0)", "IN (1)"]))


=== CatBoost (TEST) ===
Accuracy: 0.85
Balanced Acc: 0.6138
F1 (IN): 0.9153
ROC-AUC: 0.8181

Confusion Matrix:
              Pred OUT (0)  Pred IN (1)
True OUT (0)             4           10
True IN (1)              5           81

Classification Report:
              precision    recall  f1-score   support

     OUT (0)       0.44      0.29      0.35        14
      IN (1)       0.89      0.94      0.92        86

    accuracy                           0.85       100
   macro avg       0.67      0.61      0.63       100
weighted avg       0.83      0.85      0.84       100



#### Metrics

In [43]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# --- TRAIN split (fit set)
p_tr = best_cat.predict_proba(X_train)[:, 1]
pred_tr = (p_tr >= best_thr).astype(int)

# --- VAL split (threshold-tuning / validation set you used to pick best_thr)
# If you already have val_proba from the tuning loop, reuse it; otherwise compute it.
# val_proba = best_cat.predict_proba(X_val)[:, 1]
pred_va = (val_proba >= best_thr).astype(int)

# --- TEST (2024+)
p_te = test_proba  # already computed: best_cat.predict_proba(X_test)[:, 1]
pred_te = test_pred  # already computed with best_thr

def r4(x):
    return float(f"{x:.4f}")

print(f"\n=== QUICK METRICS (0=IN, 1=OUT) @ thr={best_thr:.3f} ===")
print("TRAIN  acc:", r4(accuracy_score(y_train, pred_tr)),
      "bal_acc:", r4(balanced_accuracy_score(y_train, pred_tr)),
      "F1_OUT:", r4(f1_score(y_train, pred_tr, pos_label=1)))

print("VAL    acc:", r4(accuracy_score(y_val, pred_va)),
      "bal_acc:", r4(balanced_accuracy_score(y_val, pred_va)),
      "F1_OUT:", r4(f1_score(y_val, pred_va, pos_label=1)))

print("TEST   acc:", r4(accuracy_score(y_test, pred_te)),
      "bal_acc:", r4(balanced_accuracy_score(y_test, pred_te)),
      "F1_OUT:", r4(f1_score(y_test, pred_te, pos_label=1)))


=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.470 ===
TRAIN  acc: 0.9489 bal_acc: 0.957 F1_OUT: 0.9668
VAL    acc: 0.8316 bal_acc: 0.7215 F1_OUT: 0.8959
TEST   acc: 0.85 bal_acc: 0.6138 F1_OUT: 0.9153


In [44]:
cb_importance = best_cat.get_feature_importance(type="PredictionValuesChange")

imp_df = (
    pd.DataFrame({"feature": X_train.columns, "importance": cb_importance})
      .sort_values("importance", ascending=False)
      .reset_index(drop=True)
)

print("Total features with non-zero importance:", int((imp_df["importance"] > 0).sum()))
display(imp_df.head(40))

Total features with non-zero importance: 36


,feature,importance
0,NFCI__d1__z8,10.751258
1,NFCI__d1,9.009242
2,US_10Y_Treasury__lvl__rollstd52,5.562926
3,US_2Y_Treasury__lvl__rollstd8,5.352052
4,Building_Permits__pct1__rollstd8,4.629847
5,US_3M_Treasury__d1,4.388282
6,RV_30D__d1,4.047630
7,sell_jewelry__lvl,3.831210
8,WTI_Crude_Oil__pct52,3.788207
9,Consumer_Sentiment__pct1__rollstd8,3.647738


### Data extraction

In [45]:
# =====================================================
# DF for trading sim (2024+ only) + simple checks + save
# Model: CatBoost (best_cat)
# =====================================================

# Build df (2024+)
df_trading = df.loc[df.index >= CUTOFF_DATE].copy()

# Sanity: X_test index must match df_trading index (same dates, same order)
if not df_trading.index.equals(X_test.index):
    print("WARNING: df_trading.index != X_test.index")
    print("df_trading:", df_trading.index.min(), "->", df_trading.index.max(), "n=", len(df_trading))
    print("X_test    :", X_test.index.min(),     "->", X_test.index.max(),     "n=", len(X_test))
    missing_in_df = X_test.index.difference(df_trading.index)
    missing_in_X  = df_trading.index.difference(X_test.index)
    print("Missing in df_trading (should be 0):", len(missing_in_df))
    print("Missing in X_test (should be 0):", len(missing_in_X))
    if len(missing_in_df) > 0: print("Example missing_in_df:", missing_in_df[:5].tolist())
    if len(missing_in_X)  > 0: print("Example missing_in_X :", missing_in_X[:5].tolist())

# Add preds (aligned by index)
# NOTE:
# - test_proba: P(OUT=1) from best_cat.predict_proba(X_test)[:, 1]
# - test_pred : (test_proba >= best_thr).astype(int)
df_trading["p_out"] = pd.Series(test_proba, index=X_test.index)
df_trading["pred_out"] = pd.Series(test_pred, index=X_test.index)
df_trading["y_out_true"] = pd.Series(y_test, index=X_test.index)

# NaN checks (just the important columns)
nan_counts = df_trading[["p_out", "pred_out", "y_out_true"]].isna().sum()
print("\nNaNs in key cols:\n", nan_counts)

# quick assertion-like prints
print("\nRows in df_trading:", len(df_trading))
print("Pred rows (X_test):", len(X_test))
print("All key cols non-null? ->", (nan_counts.sum() == 0))

# Save
out_path = "../predictions/catboost_preds.csv"
df_trading.to_csv(out_path)
print("\nSaved ->", out_path)


NaNs in key cols:
 p_out         0
pred_out      0
y_out_true    0
dtype: int64

Rows in df_trading: 100
Pred rows (X_test): 100
All key cols non-null? -> True

Saved -> ../predictions/catboost_preds.csv


In [46]:
# =====================================================
# FEATURE IMPORTANCE (all features, sorted)
# =====================================================
imp_all = (
    pd.DataFrame({"feature": X_train.columns, "importance": best_cat.get_feature_importance()})
      .sort_values("importance", ascending=False)
      .reset_index(drop=True)
)
imp_all["importance"] = imp_all["importance"].round(4)
print(imp_all.to_string(index=False))

                              feature  importance
                         NFCI__d1__z8     10.7513
                             NFCI__d1      9.0092
      US_10Y_Treasury__lvl__rollstd52      5.5629
        US_2Y_Treasury__lvl__rollstd8      5.3521
     Building_Permits__pct1__rollstd8      4.6298
                   US_3M_Treasury__d1      4.3883
                           RV_30D__d1      4.0476
                    sell_jewelry__lvl      3.8312
                 WTI_Crude_Oil__pct52      3.7882
   Consumer_Sentiment__pct1__rollstd8      3.6477
     credit_card_debt__lvl__rollstd52      3.4638
                        NFCI__lvl__z8      3.3974
                 resume_template__d26      3.1917
      Industrial_Production__pct1__z8      2.6054
               Building_Permits__pct8      2.5255
                   Natural_Gas__pct52      2.5090
              CPI_YoY__lvl__rollstd52      2.2845
                 Gold__pct1__rollstd8      2.1552
             Sector_Rotation__lvl__L4      2.1541
